<a href="https://colab.research.google.com/github/aapaul29/Distance-Recognition-SML1-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SML Project 1
This notebook works in both **VS Code (local)** and **Google Colab**.

## 1. Environment Detection & Setup

In [1]:
import sys
import os

IN_COLAB = 'google.colab' in sys.modules
print(f"Running in {'Google Colab' if IN_COLAB else 'VS Code / local'}")

Running in VS Code / local


## 2. Google Colab: Mount Drive & Navigate to Project
This cell only runs on Colab. Adjust `COLAB_PROJECT_PATH` to where you uploaded the project folder in your Google Drive.

In [2]:
# ---------------------------------------------------------------
# COLAB ONLY — skip if running locally
# Adjust this path to match your Google Drive folder structure.
COLAB_PROJECT_PATH = "/content/drive/MyDrive/Aaryan/Uni/BS Mech Eng ETH Zurich/2nd year/Semester 4/Stochastics & Machine Learning/Distance-Recognition-SML1-"
# ---------------------------------------------------------------

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    %cd $COLAB_PROJECT_PATH
    print(f"Working directory: {os.getcwd()}")

    # Unzip data if not already extracted
    data_zip = os.path.join(COLAB_PROJECT_PATH, "data.zip")
    data_dir = os.path.join(COLAB_PROJECT_PATH, "data")
    if os.path.exists(data_zip) and not os.path.exists(data_dir):
        print("Unzipping data...")
        os.makedirs(data_dir, exist_ok=True)
        !unzip -q $data_zip -d $data_dir
        print("Done.")
    else:
        print("Data directory already exists or no zip found — skipping unzip.")
else:
    print("Local environment — no mounting needed.")

Local environment — no mounting needed.


## 3. Install / Verify Dependencies

In [3]:
# On Colab, install any missing packages from requirements.txt.
# Locally, this is a no-op if your conda env is already set up.
if IN_COLAB:
    !pip install -q -r requirements.txt

## 4. Auto-reload & Imports

In [4]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from utils import load_config, load_dataset, load_test_dataset, print_results, save_results

# Add further sklearn imports here as needed.
# Note: SVRs are NOT allowed in this project.

## 5. Load Config & Data

In [5]:
config = load_config()

images, distances = load_dataset(config, split="train")
print(f"[INFO]: Dataset loaded with {len(images)} samples.")
print(f"[INFO]: Feature dimensionality: {images.shape[1]}")

[INFO]: Configs are loaded with: 
 {'data_dir': PosixPath('data'), 'load_rgb': True, 'downsample_factor': 3}
[INFO]: Dataset loaded with 3000 samples.
[INFO]: Feature dimensionality: 30000


## 6. Your Implementation
Implement preprocessing, model training, and evaluation below.

In [6]:
# ═══════════════════════════════════════════════════════════════
# OVERNIGHT: ds=3, RGB, proven features, stacking + clf, 5 seeds + retrain
# ═══════════════════════════════════════════════════════════════
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import (HistGradientBoostingRegressor, ExtraTreesRegressor,
                              StackingRegressor)
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.base import BaseEstimator, RegressorMixin
from scipy.ndimage import sobel, uniform_filter
import numpy as np, time

ds = config["downsample_factor"]
H = W = 300 // ds
C = 3 if config["load_rgb"] else 1
print(f"Image size: {H}x{W}, channels: {C}")

# ─── To grayscale ───
def to_gray(X):
    if C == 3:
        return X.reshape(-1, H, W, 3).mean(axis=3).reshape(len(X), -1)
    return X

# ─── LBP ───
def extract_lbp(img_2d):
    lbp = np.zeros_like(img_2d, dtype=np.uint8)
    center = img_2d[1:-1, 1:-1]
    for i, n in enumerate([
        img_2d[0:-2,0:-2], img_2d[0:-2,1:-1], img_2d[0:-2,2:],
        img_2d[1:-1,2:], img_2d[2:,2:], img_2d[2:,1:-1],
        img_2d[2:,0:-2], img_2d[1:-1,0:-2]
    ]):
        lbp[1:-1,1:-1] += (n >= center).astype(np.uint8) * (2**i)
    hist, _ = np.histogram(lbp, bins=32, range=(0,256))
    return hist.astype(np.float32) / (hist.sum() + 1e-8)

# ─── Feature extraction (works for any H, grayscale + RGB features) ───
def extract_features(X_raw, H, W, C):
    n_strips = max(10, H // 5)  # adapt strips to image size
    all_feat = []
    for img in X_raw:
        if C == 3:
            img_rgb = img.reshape(H, W, 3).astype(np.float32)
            gray = img_rgb.mean(axis=2)
            r, g, b = img_rgb[:,:,0], img_rgb[:,:,1], img_rgb[:,:,2]
        else:
            gray = img.reshape(H, W).astype(np.float32)

        w = np.linspace(0.3, 1.0, H).reshape(-1, 1)
        img_w = gray * w
        feats = []

        # Strips
        sh = H // n_strips
        means = []
        for i in range(n_strips):
            s = img_w[i*sh:(i+1)*sh, :]
            m = s.mean()
            means.append(m)
            feats.extend([m, s.std(), s.min(), s.max()])
        for i in range(1, n_strips):
            feats.append(means[i] - means[i-1])

        # Spatial pyramid
        for gs in [2, 4, 8]:
            if H % gs != 0: continue
            gh, gw = H // gs, W // gs
            for i in range(gs):
                for j in range(gs):
                    c = img_w[i*gh:(i+1)*gh, j*gw:(j+1)*gw]
                    feats.extend([c.mean(), c.std()])

        # Sobel
        sx = sobel(img_w, axis=1)
        sy = sobel(img_w, axis=0)
        feats.extend([sx.mean(), sx.var(), sy.mean(), sy.var()])
        hsx, _ = np.histogram(sx, bins=16)
        hsy, _ = np.histogram(sy, bins=16)
        feats.extend((hsx/(hsx.sum()+1e-8)).tolist())
        feats.extend((hsy/(hsy.sum()+1e-8)).tolist())

        # Floor scan
        grad = np.abs(np.diff(gray, axis=0))
        for t in [5, 10, 15, 20, 25]:
            rows = np.where(grad > t, np.arange(H-1).reshape(-1,1), -1).max(axis=0)
            v = np.where(rows >= 0, rows / H, 0).astype(np.float32)
            valid = v[v > 0]
            if len(valid) > 0:
                feats.extend([np.median(valid), np.percentile(valid,25),
                              np.percentile(valid,75), np.mean(valid), np.std(valid)])
            else:
                feats.extend([0,0,0,0,0])

        # Strip gradients (8)
        sh8 = H // 8
        for i in range(8):
            s = img_w[i*sh8:(i+1)*sh8, :]
            mag = np.sqrt(sobel(s,1)**2 + sobel(s,0)**2)
            feats.extend([mag.mean(), mag.var()])

        # Global stats
        hg, _ = np.histogram(gray, bins=16, range=(0,255))
        feats.extend((hg/(hg.sum()+1e-8)).tolist())
        for p in [5,10,25,50,75,90,95]:
            feats.append(np.percentile(gray, p))
        feats.append(gray[H//2:].mean() / (gray[:H//2].mean()+1e-8))
        feats.extend([gray.mean(), gray.std()])

        # LBP
        feats.extend(extract_lbp(gray).tolist())

        # RGB features
        if C == 3:
            for ch in [r, g, b]:
                feats.extend([ch.mean(), ch.std()])
            feats.append(r.mean()/(g.mean()+1e-8))
            feats.append(r.mean()/(b.mean()+1e-8))
            feats.append(g.mean()/(b.mean()+1e-8))
            for ch in [r, g, b]:
                feats.append(ch[H//2:].mean()/(ch[:H//2].mean()+1e-8))
            for ch in [r, g, b]:
                hc, _ = np.histogram(ch, bins=16, range=(0,255))
                feats.extend((hc/(hc.sum()+1e-8)).tolist())
            sh10 = H // 10
            for ch in [r, g, b]:
                for i in range(10):
                    feats.append(ch[i*sh10:(i+1)*sh10].mean())

        all_feat.append(np.array(feats, dtype=np.float32))
    return np.array(all_feat)

# ─── Augmentation (RGB-safe: no blur) ───
def augment(X, y):
    X_aug, y_aug = [X], [y]
    X_aug.append(X.reshape(-1,H,W,C)[:,:,::-1,:].reshape(len(X),-1)); y_aug.append(y)
    X_aug.append(np.clip(X * 0.85, 0, 255)); y_aug.append(y)
    X_aug.append(np.clip(X * 1.15, 0, 255)); y_aug.append(y)
    rng = np.random.RandomState(42)
    X_aug.append(np.clip(X + rng.normal(0, 3, X.shape), 0, 255)); y_aug.append(y)
    # Contrast
    means = X.mean(axis=1, keepdims=True)
    X_aug.append(np.clip((X - means)*1.3 + means, 0, 255)); y_aug.append(y)
    return np.vstack(X_aug), np.concatenate(y_aug)

# ─── Split ───
X_train_raw, X_val_raw, y_train_raw, y_val = train_test_split(
    images, distances, test_size=0.15, random_state=42
)

t0 = time.time()
X_train_aug, y_train_aug = augment(X_train_raw, y_train_raw)
print(f"Augmented: {len(X_train_aug)} samples ({time.time()-t0:.1f}s)")

# PCA on grayscale
X_train_gray = to_gray(X_train_raw)
w_pca = np.tile(np.linspace(0.3, 1.0, H).reshape(1,-1), (1, W))
pca_full = PCA(n_components=min(300, H*W-1), random_state=42)
pca_full.fit(X_train_gray * w_pca)

X_bot = X_train_gray.reshape(-1,H,W)[:,H//2:,:].reshape(len(X_train_raw),-1)
pca_bot = PCA(n_components=min(100, X_bot.shape[1]-1), random_state=42)
pca_bot.fit(X_bot)
print(f"PCA: {pca_full.explained_variance_ratio_.sum()*100:.1f}% ({pca_full.n_components_} comp) + {pca_bot.explained_variance_ratio_.sum()*100:.1f}% ({pca_bot.n_components_} comp)")

def build_features(X_raw, pca_f, pca_b):
    Xg = to_gray(X_raw)
    p1 = pca_f.transform(Xg * w_pca)
    p2 = pca_b.transform(Xg.reshape(-1,H,W)[:,H//2:,:].reshape(len(X_raw),-1))
    hand = extract_features(X_raw, H, W, C)
    return np.hstack([p1, p2, hand])

t0 = time.time()
print("Extracting features...")
X_train_feat = build_features(X_train_aug, pca_full, pca_bot)
X_val_feat = build_features(X_val_raw, pca_full, pca_bot)
print(f"  Done ({time.time()-t0:.1f}s), shape: {X_train_feat.shape}")

scaler = StandardScaler()
X_train_feat = scaler.fit_transform(X_train_feat)
X_val_feat = scaler.transform(X_val_feat)

Image size: 100x100, channels: 3
Augmented: 15300 samples (1.9s)
PCA: 96.4% (300 comp) + 93.7% (100 comp)
Extracting features...
  Done (50.5s), shape: (15300, 764)


In [7]:
# ─── HGB Classifier wrapper ───
from sklearn.ensemble import HistGradientBoostingClassifier

class HGBClf(BaseEstimator, RegressorMixin):
    def __init__(self, n_bins=30, **kwargs):
        self.n_bins = n_bins
        self.kwargs = kwargs
    def fit(self, X, y):
        edges = np.linspace(y.min()-0.01, y.max()+0.01, self.n_bins+1)
        self.centers_ = (edges[:-1] + edges[1:]) / 2
        yb = np.clip(np.digitize(y, edges)-1, 0, self.n_bins-1)
        self.clf_ = HistGradientBoostingClassifier(**self.kwargs)
        self.clf_.fit(X, yb)
        self.cc_ = self.centers_[self.clf_.classes_]
        return self
    def predict(self, X):
        return self.clf_.predict_proba(X) @ self.cc_

def build_stacking(seed=0):
    return StackingRegressor(
        estimators=[
            ("clf", HGBClf(n_bins=30, max_iter=800, max_depth=8, learning_rate=0.05,
                           min_samples_leaf=10, l2_regularization=0.1, random_state=seed)),
            ("hgb_mae", HistGradientBoostingRegressor(
                max_iter=800, max_depth=8, learning_rate=0.03,
                min_samples_leaf=20, loss='absolute_error',
                l2_regularization=0.1, random_state=seed)),
            ("hgb_mse", HistGradientBoostingRegressor(
                max_iter=600, max_depth=6, learning_rate=0.05,
                min_samples_leaf=30, loss='squared_error',
                l2_regularization=0.1, random_state=seed)),
            ("et", ExtraTreesRegressor(
                n_estimators=500, max_depth=25, n_jobs=-1, random_state=seed)),
            ("ridge", Ridge(alpha=1.0)),
            ("knn", KNeighborsRegressor(n_neighbors=15, weights='distance')),
        ],
        final_estimator=Ridge(alpha=10.0),
        cv=5, n_jobs=-1
    )

# ─── Train 5-seed ensemble ───
print("Training 5-seed stacking ensemble...")
print("Go to sleep - this takes ~5h\n")
models = []
val_preds = []

for seed in range(5):
    t0 = time.time()
    print(f"  Seed {seed}...")
    m = build_stacking(seed)
    m.fit(X_train_feat, y_train_aug)
    p = m.predict(X_val_feat)
    models.append(m)
    val_preds.append(p)
    print(f"    MAE: {mean_absolute_error(y_val, p)*100:.2f} cm ({time.time()-t0:.0f}s)")

val_pred = np.mean(val_preds, axis=0)
print(f"\nEnsemble MAE (5 seeds): {mean_absolute_error(y_val, val_pred)*100:.2f} cm")

Training 5-seed stacking ensemble...
Go to sleep - this takes ~5h

  Seed 0...
    MAE: 11.65 cm (553s)
  Seed 1...
    MAE: 11.81 cm (555s)
  Seed 2...
    MAE: 11.56 cm (558s)
  Seed 3...
    MAE: 11.47 cm (2241s)
  Seed 4...
    MAE: 11.41 cm (3535s)

Ensemble MAE (5 seeds): 11.44 cm


In [8]:
# Evaluation
print("=== Validation Set ===")
print_results(y_val, val_pred)

train_pred = np.mean([m.predict(X_train_feat) for m in models], axis=0)
print("\n=== Training Set ===")
print_results(y_train_aug, train_pred)
print(f"\nOverfitting gap: {(mean_absolute_error(y_val, val_pred) - mean_absolute_error(y_train_aug, train_pred))*100:.2f} cm")

=== Validation Set ===
MAE: 11.435
R2: 84.264

=== Training Set ===
MAE: 4.275
R2: 97.794

Overfitting gap: 7.16 cm


## 7. Predict on Test Set & Save

In [9]:
# ─── Retrain on ALL 3000 samples + predict test ───
print("="*60)
print("RETRAINING ON ALL 3000 SAMPLES")
print("="*60)

# Refit PCA on all data
X_all_gray = to_gray(images)
pca_full_f = PCA(n_components=min(300, H*W-1), random_state=42)
pca_full_f.fit(X_all_gray * w_pca)
X_bot_all = X_all_gray.reshape(-1,H,W)[:,H//2:,:].reshape(len(images),-1)
pca_bot_f = PCA(n_components=min(100, X_bot_all.shape[1]-1), random_state=42)
pca_bot_f.fit(X_bot_all)

# Augment all
X_all_aug, y_all_aug = augment(images, distances)
print(f"Full augmented: {len(X_all_aug)} samples")

# Features
print("Extracting features for all data...")
t0 = time.time()
X_all_feat = build_features(X_all_aug, pca_full_f, pca_bot_f)
print(f"  Done ({time.time()-t0:.1f}s)")
scaler_f = StandardScaler()
X_all_feat = scaler_f.fit_transform(X_all_feat)

# Train final models
print("Training final 5-seed ensemble...")
models_final = []
for seed in range(5):
    t0 = time.time()
    print(f"  Seed {seed}...")
    m = build_stacking(seed)
    m.fit(X_all_feat, y_all_aug)
    models_final.append(m)
    print(f"    Done ({time.time()-t0:.0f}s)")

# Test prediction
print("Processing test images...")
test_raw = np.array(load_test_dataset(config))
X_test_feat = build_features(test_raw, pca_full_f, pca_bot_f)
X_test_feat = scaler_f.transform(X_test_feat)

test_pred = np.mean([m.predict(X_test_feat) for m in models_final], axis=0)
test_pred = np.clip(test_pred, 0.1, 4.0)

save_results(test_pred)
print(f"\nSaved! Range: {test_pred.min():.3f} - {test_pred.max():.3f} m")
print("DONE - submit prediction.csv to Kaggle!")

RETRAINING ON ALL 3000 SAMPLES
Full augmented: 18000 samples
Extracting features for all data...
  Done (60.0s)
Training final 5-seed ensemble...
  Seed 0...
    Done (7827s)
  Seed 1...
    Done (5917s)
  Seed 2...
    Done (2562s)
  Seed 3...
    Done (1305s)
  Seed 4...
    Done (2373s)
Processing test images...

Saved! Range: 0.313 - 2.812 m
DONE - submit prediction.csv to Kaggle!
